# f_new6_to_4chip -- External Generalisation Result Table

Loads `08_cross_dataset_predict_new_chip.py`'s saved `predictions_final_6_new_*.joblib`
files (produced by `slurm_jobs/f_new6_to_4chip.sh`) and formats them into a LaTeX table:
one row per training-time strategy, grouped into Baseline / Training time strategy /
+ Latent Space Alignment, one column per external test chip plus a Macro Avg
(mean $\pm$ std) column, bold = best value in that column across all rows.

**Not LOFO** -- the model is trained once on all 6 `final_6_new` chips
(`--train_full_only`, no per-chip holdout) and evaluated externally against 4
*different* chips (`final_4_chip_clean_nn`) that were never seen in training at all.
So this table has 4 test-chip columns, not 6, and says "External Test Chip" instead of
"Held-Out Chip" -- there's nothing held out here, it's genuine cross-dataset transfer.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import joblib

try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        os.chdir(os.path.dirname(os.path.dirname(_nb)))
except Exception:
    pass

%load_ext autoreload
%autoreload 2
import config

print("CWD:", os.getcwd())

## Config -- mirrors `slurm_jobs/f_new6_to_4chip.sh` exactly

In [ ]:
EXP_FOLDER = config.FINAL_EXP_FOLDER  # plain POC_DDM_final, not nc_subtract -- matches the training job
TRAIN_GROUP = "final_6_new"           # what the model was trained on -- fixes the predictions_* filename prefix
CURVE_TYPE = "ori_curve_sg_p4_norm"

TEST_CHIPS = [
    "D20260806_E00_C00_F4500KHz_U_DDM_01_06",
    "D20260807_E00_C00_F4500KHz_U_DDM_02_07",
    "D20260808_E00_C00_F4500KHz_U_DDM_03_01",
    "D20260810_E00_C00_F4500KHz_U_DDM_04_01",
]
CHIP_LABELS = {
    "D20260806_E00_C00_F4500KHz_U_DDM_01_06": "Chip 01",
    "D20260807_E00_C00_F4500KHz_U_DDM_02_07": "Chip 02",
    "D20260808_E00_C00_F4500KHz_U_DDM_03_01": "Chip 03",
    "D20260810_E00_C00_F4500KHz_U_DDM_04_01": "Chip 04",
}

# (model_key, display_name) -- mirrors slurm_jobs/f_new6_to_4chip.sh's --array=0-4 exactly.
# cnn_gru_dual_attn_recon is BOTH the Baseline row and the anchor for the
# "+ Latent Space Alignment" section's first row (same model, pc_recenter on vs off).
MODELS = [
    ("cnn_gru_dual_attn_recon",         "CNN BiGRU + Spatial Attn"),
    ("cnn_gru_dual_attn_recon_dann",    "+ DANN"),
    ("cnn_gru_dual_attn_recon_supcon3", "+ SupCon"),
    ("cnn_gru_dual_attn_recon_aug",     "+ Temporal Aug"),
    ("cnn_gru_dual_attn_recon_mtl",     "+ MTL"),
]

## Load predictions + compute accuracy

`ground_truth()` is the exact pattern `notebooks/20260817-cross_dataset_confusion_matrices.ipynb`
already uses: map each pixel's raw well index to a label via `config.LABEL_MAPPINGS[chip_name]`,
then reconcile against the model's own `class_names` vocabulary (handles e.g. `NC` vs
`NC-ALL` naming variants).

In [ ]:
def ground_truth(chip_name, Y_well_raw, class_names):
    mapping = config.LABEL_MAPPINGS[chip_name]
    y_true = np.array([mapping.get(w, w) for w in Y_well_raw])
    if class_names:
        cn = list(class_names)
        y_true = np.array([next((c for c in cn if y == c or y.startswith(c + '-') or c.startswith(y + '-')), y)
                           for y in y_true])
    return y_true


def load_accuracy(chip_name, model_key, pc_recenter):
    tag = "_pc_recenter" if pc_recenter else ""
    path = (Path(EXP_FOLDER) / chip_name
            / f"predictions_{TRAIN_GROUP}_{CURVE_TYPE}_{model_key}{tag}.joblib")
    if not path.exists():
        return None, path
    d = joblib.load(path)
    y_true = ground_truth(chip_name, d["Y_well_raw"], d["class_names"])
    y_pred = np.asarray(d["pred_labels"])
    acc = float(np.mean(y_true == y_pred)) * 100
    return acc, path


# acc[model_key][pc_recenter][chip_name] = accuracy or None
acc = {m: {False: {}, True: {}} for m, _ in MODELS}
missing = []
for model_key, _ in MODELS:
    for pc_recenter in (False, True):
        for chip_name in TEST_CHIPS:
            a, path = load_accuracy(chip_name, model_key, pc_recenter)
            acc[model_key][pc_recenter][chip_name] = a
            if a is None:
                missing.append(path)

total = len(MODELS) * 2 * len(TEST_CHIPS)
if missing:
    print(f"[!] {len(missing)} / {total} prediction files not found yet -- "
          f"job/predictions still pending. Missing cells will show as \'--\' in the table below.")
    for p in missing[:10]:
        print("   ", p)
    if len(missing) > 10:
        print(f"    ... and {len(missing) - 10} more")
else:
    print("All prediction files found.")

## Build the LaTeX table

Bold = best value in that column across ALL 10 rows (both sections combined) --
matches the target format, where e.g. a chip's best score can come from the
"+ Latent Space Alignment" section rather than the baseline.

In [ ]:
def fmt_cell(v, is_best):
    if v is None:
        return "--"
    s = f"{v:.2f}"
    return rf"\textbf{{{s}}}" if is_best else s


def macro_avg(values):
    vals = [v for v in values if v is not None]
    if not vals:
        return None, None
    return float(np.mean(vals)), float(np.std(vals))


def build_table(caption, label):
    # (section_header_or_None, display_name, model_key, pc_recenter, use_quad).
    # use_quad marks "+variant" rows -- independent of whether a section header
    # prints above this row (the baseline/re-stated-baseline rows never get
    # \quad even though they open a section; "Training time strategy"'s rows
    # all get \quad even though the first one also opens a section).
    rows = [("Baseline", MODELS[0][1], MODELS[0][0], False, False)]
    rows += [("Training time strategy" if i == 0 else None, disp, key, False, True)
             for i, (key, disp) in enumerate(MODELS[1:])]
    rows += [("+ Latent Space Alignment", MODELS[0][1], MODELS[0][0], True, False)]
    rows += [(None, disp, key, True, True) for key, disp in MODELS[1:]]

    col_values = {c: [] for c in TEST_CHIPS}
    macro_values = []
    for _, _, model_key, pc_recenter, _ in rows:
        for c in TEST_CHIPS:
            col_values[c].append(acc[model_key][pc_recenter][c])
        mean, _std = macro_avg([acc[model_key][pc_recenter][c] for c in TEST_CHIPS])
        macro_values.append(mean)

    col_best = {c: max([x for x in v if x is not None], default=None) for c, v in col_values.items()}
    macro_best = max([m for m in macro_values if m is not None], default=None)

    n_chips = len(TEST_CHIPS)
    col_spec = "l " + "r" * n_chips + " r"
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"    \centering")
    lines.append(rf"    \caption{{{caption}}}")
    lines.append(rf"    \label{{{label}}}")
    lines.append(r"    \footnotesize")
    lines.append(r"    \setlength{\tabcolsep}{1.5pt}")
    lines.append(rf"    \begin{{tabular}}{{@{{}}{col_spec}@{{}}}}")
    lines.append(r"    \toprule")
    lines.append(rf"    & \multicolumn{{{n_chips}}}{{c}}{{\textbf{{External Test Chip}}}} & \\")
    lines.append(rf"    \cmidrule(lr){{2-{n_chips + 1}}}")
    header = " & ".join(rf"\textbf{{{CHIP_LABELS[c]}}}" for c in TEST_CHIPS)
    lines.append(rf"    \textbf{{Model}} & {header} & \textbf{{Macro Avg}} \\")
    lines.append(r"    \midrule")

    prev_section = None
    for section, display_name, model_key, pc_recenter, use_quad in rows:
        if section is not None:
            if prev_section is not None:
                lines.append(r"    \addlinespace")
            lines.append(rf"    \multicolumn{{{n_chips + 1}}}{{l}}{{\textbf{{{section}}}}} \\")
            prev_section = section
        row_name = rf"\quad {display_name}" if use_quad else display_name

        row_cells = []
        for c in TEST_CHIPS:
            v = acc[model_key][pc_recenter][c]
            is_best = col_best[c] is not None and v == col_best[c]
            row_cells.append(fmt_cell(v, is_best))
        cells_str = " & ".join(row_cells)

        mean, std = macro_avg([acc[model_key][pc_recenter][c] for c in TEST_CHIPS])
        if mean is None:
            macro_str = "--"
        else:
            is_best = macro_best is not None and mean == macro_best
            s = rf"{mean:.2f} $\pm$ {std:.2f}"
            macro_str = rf"\textbf{{{s}}}" if is_best else s

        lines.append(rf"    {row_name} & {cells_str} & {macro_str} \\")

    lines.append(r"    \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


N = len(TEST_CHIPS)
caption = (
    rf"Accuracy on {N} external test chips (never used in training) for the baseline, "
    r"each training-time strategy, and the same conditions with the inference-time "
    r"PC-anchored latent-space alignment additionally applied. The model in every row "
    r"is trained once on all 6 chips of the training group, with no held-out chip. "
    r"The Macro Avg column reports the mean $\pm$ standard deviation across the "
    rf"{N} test chips, and all values are percentages. Bold marks the best result in "
    r"each column across all listed conditions."
)
latex = build_table(caption, "tab:f_new6_to_4chip_result")
print(latex)

## Save to file

In [ ]:
out_path = Path(EXP_FOLDER) / "cross_dataset_cv" / TRAIN_GROUP / "f_new6_to_4chip_table.tex"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(latex)
print(f"[SAVED] {out_path}")